In [1]:
import os
import time
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import (
    Layer, Input, Dense, LayerNormalization, MultiHeadAttention,
    Dropout, GlobalAveragePooling1D, Conv1D, TimeDistributed
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# =========================
# Reproducibility
# =========================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# Dataset parameters
# =========================

EPOCHS = 50
N_SPLITS = 5

SEQ_LEN = 2560
CHANNELS = 27
N_CLASSES = 2

DATASET_NAME = "Raw"

# Best hyperparameters (Optuna)
best_params = {
    'patch_size': 128,
    'projection_dim': 128,
    'num_heads': 8,
    'transformer_layers': 4,
    'ff_dim': 128,
    'dropout_rate': 0.24669,
    'learning_rate': 0.000304,
    'batch_size': 32
}

# =========================
# Load metadata
# =========================

df = pd.read_csv("dataset_metadata_raw.csv")

print(f"Total samples: {len(df)}")
print(f"Total subjects: {df['subject'].nunique()}")

# =========================
# Data Generator
# =========================

class EEGGenerator(Sequence):

    def __init__(self, df, batch_size=32, shuffle=True):

        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.df))

        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):

        batch_idx = self.indexes[idx*self.batch_size:(idx+1)*self.batch_size]
        batch_df = self.df.iloc[batch_idx]

        X, y = [], []

        for _, row in batch_df.iterrows():

            data = np.load(row["file"])

            if data.shape == (SEQ_LEN, CHANNELS):
                pass
            elif data.shape == (CHANNELS, SEQ_LEN):
                data = data.T
            else:
                raise ValueError("Unexpected shape")

            X.append(data)
            y.append(row["label"])

        X = np.array(X, dtype=np.float32)
        y = tf.keras.utils.to_categorical(y, N_CLASSES)

        return X, y

    def on_epoch_end(self):

        if self.shuffle:
            np.random.shuffle(self.indexes)

# =========================
# Patches Layer
# =========================
class Patches(Layer):

    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, signals):

        batch_size = tf.shape(signals)[0]

        patches = tf.reshape(
            signals,
            [batch_size, -1, self.patch_size, tf.shape(signals)[-1]]
        )

        return patches

    def get_config(self):
        config = super().get_config()
        config.update({
            "patch_size": self.patch_size,
        })
        return config

# =========================
# Position Embedding
# =========================

def get_sinusoidal_position_embedding(sequence_length, dim):

    position = np.arange(sequence_length)[:, np.newaxis]
    div_term = np.exp(np.arange(0, dim, 2) * -(np.log(10000.0) / dim))

    pe = np.zeros((sequence_length, dim))

    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)

    return tf.constant(pe, dtype=tf.float32)


class SinusoidalPositionEmbedding(Layer):

    def __init__(self, sequence_length, dim, **kwargs):
        super().__init__(**kwargs)

        self.sequence_length = sequence_length
        self.dim = dim

        self.pos_embedding = get_sinusoidal_position_embedding(
            sequence_length, dim
        )

    def call(self, x):

        return x + self.pos_embedding

    def get_config(self):
        config = super().get_config()
        config.update({
            "sequence_length": self.sequence_length,
            "dim": self.dim,
        })
        return config

# =========================
# CvT Model
# =========================

def build_cvt():

    patch_size = best_params['patch_size']
    projection_dim = best_params['projection_dim']
    num_heads = best_params['num_heads']
    transformer_layers = best_params['transformer_layers']
    ff_dim = best_params['ff_dim']
    dropout = best_params['dropout_rate']

    num_patches = SEQ_LEN // patch_size

    inputs = Input(shape=(SEQ_LEN, CHANNELS))

    patches = Patches(patch_size)(inputs)

    cnn_encoder = tf.keras.Sequential([

        Input(shape=(patch_size, CHANNELS)),
        Conv1D(64, 10, padding='same', activation='relu'),
        Conv1D(64, 5, padding='same', activation='relu'),
        GlobalAveragePooling1D(),
        Dense(projection_dim)

    ])

    encoded = TimeDistributed(cnn_encoder)(patches)

    encoded = SinusoidalPositionEmbedding(
        num_patches,
        projection_dim
    )(encoded)

    for _ in range(transformer_layers):

        x1 = LayerNormalization(epsilon=1e-6)(encoded)

        attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=projection_dim//num_heads,
            dropout=dropout
        )(x1, x1)

        x2 = tf.keras.layers.Add()([attention, encoded])

        x3 = LayerNormalization(epsilon=1e-6)(x2)

        x3 = Dense(ff_dim, activation="relu")(x3)
        x3 = Dropout(dropout)(x3)
        x3 = Dense(projection_dim)(x3)

        encoded = tf.keras.layers.Add()([x3, x2])

    representation = LayerNormalization(epsilon=1e-6)(encoded)

    representation = GlobalAveragePooling1D()(representation)

    representation = Dropout(0.5)(representation)

    x = Dense(128, activation="relu")(representation)

    outputs = Dense(N_CLASSES, activation="softmax")(x)

    model = Model(inputs, outputs)

    optimizer = Adam(learning_rate=best_params['learning_rate'])

    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =========================
# Results storage
# =========================

os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

all_results = []

# =========================
# Cross Subject CV
# =========================

gkf = GroupKFold(n_splits=N_SPLITS)
groups = df["subject"]

for fold, (train_idx, val_idx) in enumerate(
    gkf.split(df, df["label"], groups)
):

    print("\n====================")
    print(f"Fold {fold+1}/{N_SPLITS}")
    print("====================")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    print("Train:", len(train_df))
    print("Val:", len(val_df))

    train_gen = EEGGenerator(train_df,
                             best_params['batch_size'],
                             shuffle=True)

    val_gen = EEGGenerator(val_df,
                           best_params['batch_size'],
                           shuffle=False)

    model = build_cvt()

    checkpoint_path = f"models/cvt_{DATASET_NAME}_fold{fold+1}.h5"

    callbacks = [

        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        ),

        ModelCheckpoint(
            checkpoint_path,
            monitor="val_loss",
            save_best_only=True
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6
        )

    ]

    start = time.time()

    model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    print("Training time:",
          round(time.time()-start,2),
          "seconds")

    # =========================
    # Evaluation
    # =========================

    y_true, y_pred, y_prob = [], [], []

    for X_batch, y_batch in val_gen:

        preds = model.predict(X_batch, verbose=0)

        y_true.extend(np.argmax(y_batch, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
        y_prob.extend(preds[:,1])

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    acc = accuracy_score(y_true,y_pred)
    precision = precision_score(y_true,y_pred)
    recall = recall_score(y_true,y_pred)
    f1 = f1_score(y_true,y_pred)
    roc = roc_auc_score(y_true,y_prob)

    tn, fp, fn, tp = confusion_matrix(y_true,y_pred).ravel()
    specificity = tn/(tn+fp)

    print(f"Accuracy: {acc:.3f} | F1: {f1:.3f} | ROC-AUC: {roc:.3f}")

    all_results.append({

        "dataset":DATASET_NAME,
        "fold":fold+1,
        "accuracy":acc,
        "precision":precision,
        "recall":recall,
        "specificity":specificity,
        "f1":f1,
        "roc_auc":roc,
        "tp":tp,"tn":tn,"fp":fp,"fn":fn
    })

# =========================
# Save Results
# =========================

results_df = pd.DataFrame(all_results)

csv_path = f"results/cvt_results_{DATASET_NAME}.csv"

results_df.to_csv(csv_path,index=False)

print("\nSaved results to:",csv_path)

print("\nMean performance across folds:\n")

print(results_df.mean(numeric_only=True))

Total samples: 20792
Total subjects: 87

Fold 1/5
Train: 16480
Val: 4312
Epoch 1/50
515/515 [==============================] - 48s 78ms/step - loss: 0.7213 - accuracy: 0.5042 - val_loss: 0.6921 - val_accuracy: 0.5007 - lr: 3.0400e-04
Epoch 2/50
515/515 [==============================] - 37s 72ms/step - loss: 0.7027 - accuracy: 0.5098 - val_loss: 0.6766 - val_accuracy: 0.5907 - lr: 3.0400e-04
Epoch 3/50
515/515 [==============================] - 36s 69ms/step - loss: 0.6836 - accuracy: 0.5569 - val_loss: 0.6345 - val_accuracy: 0.6656 - lr: 3.0400e-04
Epoch 4/50
515/515 [==============================] - 35s 69ms/step - loss: 0.6343 - accuracy: 0.6383 - val_loss: 0.6148 - val_accuracy: 0.6408 - lr: 3.0400e-04
Epoch 5/50
515/515 [==============================] - 36s 69ms/step - loss: 0.6004 - accuracy: 0.6740 - val_loss: 0.5644 - val_accuracy: 0.6990 - lr: 3.0400e-04
Epoch 6/50
515/515 [==============================] - 37s 71ms/step - loss: 0.5838 - accuracy: 0.6868 - val_loss: 0.5549 -

In [2]:
import os
import time
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import (
    Layer, Input, Dense, LayerNormalization, MultiHeadAttention,
    Dropout, GlobalAveragePooling1D, Conv1D, TimeDistributed
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# =========================
# Reproducibility
# =========================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# Dataset parameters
# =========================

EPOCHS = 50
N_SPLITS = 5

SEQ_LEN = 2560
CHANNELS = 27
N_CLASSES = 2

DATASET_NAME = "Filtered_8_30Hz"

# Best hyperparameters (Optuna)
best_params = {
    'patch_size': 128,
    'projection_dim': 128,
    'num_heads': 4,
    'transformer_layers': 4,
    'ff_dim': 128,
    'dropout_rate': 0.33870,
    'learning_rate': 0.000160,
    'batch_size': 32
}

# =========================
# Load metadata
# =========================

df = pd.read_csv("dataset_metadata_filtered.csv")

print(f"Total samples: {len(df)}")
print(f"Total subjects: {df['subject'].nunique()}")

# =========================
# Data Generator
# =========================

class EEGGenerator(Sequence):

    def __init__(self, df, batch_size=32, shuffle=True):

        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.df))

        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):

        batch_idx = self.indexes[idx*self.batch_size:(idx+1)*self.batch_size]
        batch_df = self.df.iloc[batch_idx]

        X, y = [], []

        for _, row in batch_df.iterrows():

            data = np.load(row["file"])

            if data.shape == (SEQ_LEN, CHANNELS):
                pass
            elif data.shape == (CHANNELS, SEQ_LEN):
                data = data.T
            else:
                raise ValueError("Unexpected shape")

            X.append(data)
            y.append(row["label"])

        X = np.array(X, dtype=np.float32)
        y = tf.keras.utils.to_categorical(y, N_CLASSES)

        return X, y

    def on_epoch_end(self):

        if self.shuffle:
            np.random.shuffle(self.indexes)

# =========================
# Patches Layer
# =========================
class Patches(Layer):

    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, signals):

        batch_size = tf.shape(signals)[0]

        patches = tf.reshape(
            signals,
            [batch_size, -1, self.patch_size, tf.shape(signals)[-1]]
        )

        return patches

    def get_config(self):
        config = super().get_config()
        config.update({
            "patch_size": self.patch_size,
        })
        return config

# =========================
# Position Embedding
# =========================

def get_sinusoidal_position_embedding(sequence_length, dim):

    position = np.arange(sequence_length)[:, np.newaxis]
    div_term = np.exp(np.arange(0, dim, 2) * -(np.log(10000.0) / dim))

    pe = np.zeros((sequence_length, dim))

    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)

    return tf.constant(pe, dtype=tf.float32)


class SinusoidalPositionEmbedding(Layer):

    def __init__(self, sequence_length, dim, **kwargs):
        super().__init__(**kwargs)

        self.sequence_length = sequence_length
        self.dim = dim

        self.pos_embedding = get_sinusoidal_position_embedding(
            sequence_length, dim
        )

    def call(self, x):

        return x + self.pos_embedding

    def get_config(self):
        config = super().get_config()
        config.update({
            "sequence_length": self.sequence_length,
            "dim": self.dim,
        })
        return config

# =========================
# CvT Model
# =========================

def build_cvt():

    patch_size = best_params['patch_size']
    projection_dim = best_params['projection_dim']
    num_heads = best_params['num_heads']
    transformer_layers = best_params['transformer_layers']
    ff_dim = best_params['ff_dim']
    dropout = best_params['dropout_rate']

    num_patches = SEQ_LEN // patch_size

    inputs = Input(shape=(SEQ_LEN, CHANNELS))

    patches = Patches(patch_size)(inputs)

    cnn_encoder = tf.keras.Sequential([

        Input(shape=(patch_size, CHANNELS)),
        Conv1D(64, 10, padding='same', activation='relu'),
        Conv1D(64, 5, padding='same', activation='relu'),
        GlobalAveragePooling1D(),
        Dense(projection_dim)

    ])

    encoded = TimeDistributed(cnn_encoder)(patches)

    encoded = SinusoidalPositionEmbedding(
        num_patches,
        projection_dim
    )(encoded)

    for _ in range(transformer_layers):

        x1 = LayerNormalization(epsilon=1e-6)(encoded)

        attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=projection_dim//num_heads,
            dropout=dropout
        )(x1, x1)

        x2 = tf.keras.layers.Add()([attention, encoded])

        x3 = LayerNormalization(epsilon=1e-6)(x2)

        x3 = Dense(ff_dim, activation="relu")(x3)
        x3 = Dropout(dropout)(x3)
        x3 = Dense(projection_dim)(x3)

        encoded = tf.keras.layers.Add()([x3, x2])

    representation = LayerNormalization(epsilon=1e-6)(encoded)

    representation = GlobalAveragePooling1D()(representation)

    representation = Dropout(0.5)(representation)

    x = Dense(128, activation="relu")(representation)

    outputs = Dense(N_CLASSES, activation="softmax")(x)

    model = Model(inputs, outputs)

    optimizer = Adam(learning_rate=best_params['learning_rate'])

    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =========================
# Results storage
# =========================

os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

all_results = []

# =========================
# Cross Subject CV
# =========================

gkf = GroupKFold(n_splits=N_SPLITS)
groups = df["subject"]

for fold, (train_idx, val_idx) in enumerate(
    gkf.split(df, df["label"], groups)
):

    print("\n====================")
    print(f"Fold {fold+1}/{N_SPLITS}")
    print("====================")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    print("Train:", len(train_df))
    print("Val:", len(val_df))

    train_gen = EEGGenerator(train_df,
                             best_params['batch_size'],
                             shuffle=True)

    val_gen = EEGGenerator(val_df,
                           best_params['batch_size'],
                           shuffle=False)

    model = build_cvt()

    checkpoint_path = f"models/cvt_{DATASET_NAME}_fold{fold+1}.h5"

    callbacks = [

        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        ),

        ModelCheckpoint(
            checkpoint_path,
            monitor="val_loss",
            save_best_only=True
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6
        )

    ]

    start = time.time()

    model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    print("Training time:",
          round(time.time()-start,2),
          "seconds")

    # =========================
    # Evaluation
    # =========================

    y_true, y_pred, y_prob = [], [], []

    for X_batch, y_batch in val_gen:

        preds = model.predict(X_batch, verbose=0)

        y_true.extend(np.argmax(y_batch, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
        y_prob.extend(preds[:,1])

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    acc = accuracy_score(y_true,y_pred)
    precision = precision_score(y_true,y_pred)
    recall = recall_score(y_true,y_pred)
    f1 = f1_score(y_true,y_pred)
    roc = roc_auc_score(y_true,y_prob)

    tn, fp, fn, tp = confusion_matrix(y_true,y_pred).ravel()
    specificity = tn/(tn+fp)

    print(f"Accuracy: {acc:.3f} | F1: {f1:.3f} | ROC-AUC: {roc:.3f}")

    all_results.append({

        "dataset":DATASET_NAME,
        "fold":fold+1,
        "accuracy":acc,
        "precision":precision,
        "recall":recall,
        "specificity":specificity,
        "f1":f1,
        "roc_auc":roc,
        "tp":tp,"tn":tn,"fp":fp,"fn":fn
    })

# =========================
# Save Results
# =========================

results_df = pd.DataFrame(all_results)

csv_path = f"results/cvt_results_{DATASET_NAME}.csv"

results_df.to_csv(csv_path,index=False)

print("\nSaved results to:",csv_path)

print("\nMean performance across folds:\n")

print(results_df.mean(numeric_only=True))

Total samples: 20792
Total subjects: 87

Fold 1/5
Train: 16480
Val: 4312
Epoch 1/50
515/515 [==============================] - 193s 368ms/step - loss: 0.7191 - accuracy: 0.5044 - val_loss: 0.6935 - val_accuracy: 0.5005 - lr: 1.6000e-04
Epoch 2/50
515/515 [==============================] - 36s 70ms/step - loss: 0.7087 - accuracy: 0.4999 - val_loss: 0.6951 - val_accuracy: 0.4995 - lr: 1.6000e-04
Epoch 3/50
515/515 [==============================] - 37s 71ms/step - loss: 0.6975 - accuracy: 0.5262 - val_loss: 0.6647 - val_accuracy: 0.5988 - lr: 1.6000e-04
Epoch 4/50
515/515 [==============================] - 37s 71ms/step - loss: 0.6602 - accuracy: 0.5981 - val_loss: 0.6601 - val_accuracy: 0.6146 - lr: 1.6000e-04
Epoch 5/50
515/515 [==============================] - 39s 76ms/step - loss: 0.6309 - accuracy: 0.6362 - val_loss: 0.6366 - val_accuracy: 0.6322 - lr: 1.6000e-04
Epoch 6/50
515/515 [==============================] - 37s 72ms/step - loss: 0.6186 - accuracy: 0.6441 - val_loss: 0.6315

In [1]:
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import (
    Layer, Input, Dense, LayerNormalization, MultiHeadAttention,
    Dropout, GlobalAveragePooling1D, Conv1D, TimeDistributed
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# =========================
# Reproducibility
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# Dataset parameters
# =========================
EPOCHS = 50
N_SPLITS = 5
SEQ_LEN = 2560
CHANNELS = 27
N_CLASSES = 2

DATASET_NAME = "Filtered_8_13Hz"
CSV_FILE = "dataset_metadata_8_13Hz.csv"

# ✅ Best hyperparameters (Optuna) for 8-13Hz
best_params = {
    'patch_size': 160,
    'projection_dim': 128,
    'num_heads': 8,
    'transformer_layers': 5,
    'ff_dim': 128,
    'dropout_rate': 0.19138,
    'learning_rate': 0.000315,
    'batch_size': 32
}

# =========================
# Load metadata
# =========================
df = pd.read_csv(CSV_FILE)
print(f"Total samples: {len(df)}")
print(f"Total subjects: {df['subject'].nunique()}")

# =========================
# Data Generator
# =========================
class EEGGenerator(Sequence):
    def __init__(self, df, batch_size=32, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indexes[idx*self.batch_size:(idx+1)*self.batch_size]
        batch_df = self.df.iloc[batch_idx]

        X, y = [], []
        for _, row in batch_df.iterrows():
            data = np.load(row["file"])
            # Handle shape dynamically
            if data.shape == (SEQ_LEN, CHANNELS):
                pass
            elif data.shape == (CHANNELS, SEQ_LEN):
                data = data.T
            else:
                raise ValueError(f"Unexpected shape: {data.shape}")
            
            X.append(data)
            y.append(row["label"])

        X = np.array(X, dtype=np.float32)
        y = tf.keras.utils.to_categorical(y, N_CLASSES)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

# =========================
# Patches Layer
# =========================
class Patches(Layer):
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, signals):
        batch_size = tf.shape(signals)[0]
        patches = tf.reshape(
            signals,
            [batch_size, -1, self.patch_size, tf.shape(signals)[-1]]
        )
        return patches

    def get_config(self):
        config = super().get_config()
        config.update({"patch_size": self.patch_size})
        return config

# =========================
# Position Embedding
# =========================
def get_sinusoidal_position_embedding(sequence_length, dim):
    position = np.arange(sequence_length)[:, np.newaxis]
    div_term = np.exp(np.arange(0, dim, 2) * -(np.log(10000.0) / dim))
    pe = np.zeros((sequence_length, dim))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return tf.constant(pe, dtype=tf.float32)

class SinusoidalPositionEmbedding(Layer):
    def __init__(self, sequence_length, dim, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = sequence_length
        self.dim = dim
        self.pos_embedding = get_sinusoidal_position_embedding(sequence_length, dim)

    def call(self, x):
        return x + self.pos_embedding

    def get_config(self):
        config = super().get_config()
        config.update({
            "sequence_length": self.sequence_length,
            "dim": self.dim,
        })
        return config

# =========================
# CvT Model
# =========================
def build_cvt():
    patch_size = best_params['patch_size']
    projection_dim = best_params['projection_dim']
    num_heads = best_params['num_heads']
    transformer_layers = best_params['transformer_layers']
    ff_dim = best_params['ff_dim']
    dropout = best_params['dropout_rate']

    num_patches = SEQ_LEN // patch_size

    inputs = Input(shape=(SEQ_LEN, CHANNELS))
    patches = Patches(patch_size)(inputs)

    cnn_encoder = tf.keras.Sequential([
        Input(shape=(patch_size, CHANNELS)),
        Conv1D(64, 10, padding='same', activation='relu'),
        Conv1D(64, 5, padding='same', activation='relu'),
        GlobalAveragePooling1D(),
        Dense(projection_dim)
    ])

    encoded = TimeDistributed(cnn_encoder)(patches)
    encoded = SinusoidalPositionEmbedding(num_patches, projection_dim)(encoded)

    for _ in range(transformer_layers):
        x1 = LayerNormalization(epsilon=1e-6)(encoded)
        attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=projection_dim//num_heads,
            dropout=dropout
        )(x1, x1)
        x2 = tf.keras.layers.Add()([attention, encoded])
        
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        x3 = Dense(ff_dim, activation="relu")(x3)
        x3 = Dropout(dropout)(x3)
        x3 = Dense(projection_dim)(x3)
        encoded = tf.keras.layers.Add()([x3, x2])

    representation = LayerNormalization(epsilon=1e-6)(encoded)
    representation = GlobalAveragePooling1D()(representation)
    representation = Dropout(0.5)(representation)
    
    x = Dense(128, activation="relu")(representation)
    outputs = Dense(N_CLASSES, activation="softmax")(x)

    model = Model(inputs, outputs)
    optimizer = Adam(learning_rate=best_params['learning_rate'])
    
    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# =========================
# Results storage
# =========================
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)
all_results = []

# =========================
# Cross Subject CV
# =========================
gkf = GroupKFold(n_splits=N_SPLITS)
groups = df["subject"]

for fold, (train_idx, val_idx) in enumerate(gkf.split(df, df["label"], groups)):
    print("\n====================")
    print(f"Fold {fold+1}/{N_SPLITS}")
    print("====================")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    print("Train samples:", len(train_df))
    print("Val samples:", len(val_df))

    train_gen = EEGGenerator(train_df, best_params['batch_size'], shuffle=True)
    val_gen = EEGGenerator(val_df, best_params['batch_size'], shuffle=False)

    model = build_cvt()

    checkpoint_path = f"models/cvt_{DATASET_NAME}_fold{fold+1}.h5"
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)
    ]

    start = time.time()
    model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )
    print("Training time:", round(time.time()-start, 2), "seconds")

    # =========================
    # Evaluation
    # =========================
    y_true, y_pred, y_prob = [], [], []
    for X_batch, y_batch in val_gen:
        preds = model.predict(X_batch, verbose=0)
        y_true.extend(np.argmax(y_batch, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
        y_prob.extend(preds[:,1])

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc = roc_auc_score(y_true, y_prob)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn/(tn+fp)

    print(f"Accuracy: {acc:.3f} | F1: {f1:.3f} | ROC-AUC: {roc:.3f}")

    all_results.append({
        "dataset": DATASET_NAME,
        "fold": fold+1,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "roc_auc": roc,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    })
    
    del model
    tf.keras.backend.clear_session()
    gc.collect()

# =========================
# Save Results
# =========================
results_df = pd.DataFrame(all_results)
csv_path = f"results/cvt_results_{DATASET_NAME}.csv"
results_df.to_csv(csv_path, index=False)

print("\n✅ Saved results to:", csv_path)
print("\nMean performance across folds:\n")

try:
    print(results_df.mean(numeric_only=True))
except Exception as e:
    print("Mean calculation skipped:", e)

Total samples: 20792
Total subjects: 87

Fold 1/5
Train samples: 16480
Val samples: 4312
Epoch 1/50
515/515 [==============================] - 50s 79ms/step - loss: 0.7256 - accuracy: 0.4963 - val_loss: 0.6926 - val_accuracy: 0.4998 - lr: 3.1500e-04
Epoch 2/50
515/515 [==============================] - 32s 61ms/step - loss: 0.7063 - accuracy: 0.5019 - val_loss: 0.6920 - val_accuracy: 0.5021 - lr: 3.1500e-04
Epoch 3/50
515/515 [==============================] - 32s 61ms/step - loss: 0.6635 - accuracy: 0.5913 - val_loss: 0.6460 - val_accuracy: 0.6143 - lr: 3.1500e-04
Epoch 4/50
515/515 [==============================] - 34s 65ms/step - loss: 0.6267 - accuracy: 0.6383 - val_loss: 0.6305 - val_accuracy: 0.6449 - lr: 3.1500e-04
Epoch 5/50
515/515 [==============================] - 32s 62ms/step - loss: 0.6135 - accuracy: 0.6492 - val_loss: 0.6163 - val_accuracy: 0.6545 - lr: 3.1500e-04
Epoch 6/50
515/515 [==============================] - 36s 69ms/step - loss: 0.6109 - accuracy: 0.6542 - va